# Imports

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from harbor.analysis.cross_docking import DockingDataModel
from importlib import reload

# Load Data

In [ ]:
posit_results = Path("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/full_cross_dock_v2_analyzed_results/")
pdf = pd.read_csv(posit_results)
pdf["Error_Lower"] = pdf["Fraction"] - pdf["CI_Lower"]
pdf["Error_Lower"] = pdf["Error_Lower"].apply(lambda x: 0 if x < 0 else x)
pdf["Error_Upper"] = pdf["CI_Upper"] - pdf["Fraction"]
pdf["Error_Upper"] = pdf["Error_Upper"].apply(lambda x: 0 if x < 0 else x)

# Plotting Params

In [ ]:
# Global configuration
sns.set_style("white")

X_VAR = "N_Reference_Structures"
Y_VAR = "Fraction"
X_LABEL = "Number of Reference Structures Available to Use \n(Log Scale)"
Y_LABEL = "Fraction of Ligands Posed \n<2Å from Reference"
QUERY_SCAFFOLD_ID = "Query_Scaffold_ID_Subset_1"
REF_SCAFFOLD_ID = "Reference_Scaffold_ID_Subset_1"
COLOR_VAR = "Reference_Split"
STYLE_VAR = "Score"
CI_LOWER = "CI_Lower"
CI_UPPER = "CI_Upper"
label_map = {
    "Reference_Split": "Dataset Split Type",
    "Score": "Scoring Method",
    "RandomSplit": "Randomly Ordered",
    "DateSplit": "Ordered by Date",
    "RMSD": "RMSD (Positive Control)",
    "POSIT_Probability": "POSIT Probability",
    "N_Reference_Structures": "Number of Randomly Chosen Reference Structures"
}
LARGE_FIG_SIZE = (12, 8)
SMALL_FIG_SIZE = (8, 6)
FONT_SIZES = {
    "xlabel": 24,
    "ylabel": 24,
    "ticks": 18,
    "legend_title": 24,
    "legend_text": 18,
}
ALPHA = 0.1

# Make Figure

In [ ]:
df = pdf[(~pdf["Reference_Split"].isna())&(pdf["PairwiseSplit"].isna())]

In [ ]:
df = df.groupby(["Reference_Split", "Score", "N_Reference_Structures"]).head(1)

In [ ]:
ALPHA = 0.3
p.figure_decorator(func=plot_filled_in_error_bars, label_map=label_map, fig_path=figpath / "dataset_split", raw_df=df)
ALPHA = 0.2

In [ ]:
def plot_filled_in_error_bars_facet_col(
    raw_df,
    x_var=X_VAR,
    y_var=Y_VAR,
    color_var=COLOR_VAR,
    style_var=STYLE_VAR,
    facet_var=STYLE_VAR,  # New parameter, defaults to None
    ci_lower=CI_LOWER,
    ci_upper=CI_UPPER,
):
    """Plot filled-in error bars with facets by specified variable"""
    # Use style_var as facet_var if none provided
    facet_var = facet_var or style_var
    style_var = style_var or color_var

    # Sort the dataframe
    raw_df = raw_df.sort_values(by=[x_var, style_var, color_var])

    # Create subplot for each facet value
    facets = raw_df[facet_var].unique()
    fig, axes = plt.subplots(1, len(facets), figsize=(LARGE_FIG_SIZE[0]*len(facets), LARGE_FIG_SIZE[1]))

    # Create color mapping
    unique_colors = sns.color_palette(n_colors=len(raw_df[color_var].unique()))
    color_map = dict(zip(sorted(raw_df[color_var].unique()), unique_colors))

    for ax, facet in zip(axes, facets):
        facet_data = raw_df[raw_df[facet_var] == facet]

        # Create fill between for each group using matched colors
        for name, group in facet_data.groupby([color_var, style_var]):
            color_name = name[0]  # First element is Score
            ax.fill_between(
                group[x_var],
                group[ci_lower],
                group[ci_upper],
                color=color_map[color_name],
                alpha=ALPHA,
            )

        # Create the line plot
        sns.lineplot(
            data=facet_data,
            x=x_var,
            y=y_var,
            hue=color_var,
            style=style_var,  # Keep style_var for line styles
            ax=ax,
            palette=color_map,
            hue_order=list(reversed(sorted(raw_df[color_var].unique()))),
            style_order=list(reversed(sorted(raw_df[style_var].unique())))
        )
        
        # Customize each subplot
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(ScalarFormatter())

        custom_ticks = [1, 5, 10, 20, 50, 100, 200, raw_df[x_var].max()]
        ax.set_xticks(custom_ticks)
        ax.set_xticklabels(custom_ticks, fontsize=FONT_SIZES["ticks"])
        ax.tick_params(axis='y', labelsize=FONT_SIZES["ticks"])

        ax.set_xlabel(X_LABEL, fontsize=FONT_SIZES["xlabel"], fontweight="bold")
        if ax == axes[0]:  # Only set ylabel for first subplot
            ax.set_ylabel(Y_LABEL, fontsize=FONT_SIZES["ylabel"], fontweight="bold")
        else:
            ax.set_ylabel("")

        ax.set_title(f"{facet}", fontsize=FONT_SIZES["xlabel"], fontweight="bold")

        # Customize legend
        legend = ax.legend()
        plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
        plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])
    return plt